In [ ]:
import time
import pandas as pd
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score, accuracy_score, precision_recall_fscore_support

from utils import *

import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

hola


In [ ]:
BASE_PATH = Path("../data")
EXP_PATH = BASE_PATH / "EXPERIMENT1"
DOMAIN_PATH = EXP_PATH / "domain_adaptation"

TRAIN_PATH = DOMAIN_PATH / "train_public_64"
VAL_PATH   = DOMAIN_PATH / "val_public_16"
TEST_PUBLIC_PATH = DOMAIN_PATH / "test"   # In-domain test
TEST_LOCAL_PATH  = EXP_PATH / "local_data"          # Out-of-domain test
RESULTS_PATH = EXP_PATH / "models"
OUTPUT_PATH = DOMAIN_PATH / "results"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32

IMG_SIZE = (224,224)
model_list = [
    "EffNetB3_CBAM",
    "ResNet101_CBAM",
    "mobilenetv2",
    "efficientnetb0",
    "mobilenetv3large",
    "densenet121",
    "efficientnetb3",
    "resnet50",
    "inceptionv3",
    "resnet101",
    "vgg16",
    "vgg19",
    "efficientnetb5"
]

NUM_CLASSES = 4

# EVALUATING DOMAIN GAP FOR CNN-MODELS. (domain_gap_finetuning.ipynb, cell 4)

In [ ]:


results = []


class_weights_array_for_loading = np.ones(NUM_CLASSES)

for backbone in model_list:
    print(f"\n🔹 Evaluando modelo: {backbone}")

    model = None # Initialize model to None for each iteration
    best_model_path = RESULTS_PATH / f"{backbone}_best_model.keras"
    if not best_model_path.exists():
        print(f"⚠️ Modelo no encontrado: {best_model_path}")
        continue # If model not found, skip to next backbone

    # Define comprehensive custom_objects
    custom_objects = {
    "weighted_categorical_crossentropy": weighted_categorical_crossentropy,
    "recall_macro": recall_macro,
    "precision_macro": precision_macro,
    "f1_macro": f1_macro
    }  

    # Load the model here if best_model_path exists
    model = tf.keras.models.load_model(
        best_model_path,
        compile=False,
        custom_objects=custom_objects
    )


    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=weighted_categorical_crossentropy(class_weights_array_for_loading),
        metrics=[f1_macro, precision_macro, recall_macro]
    )

    # Determine the correct backbone name for the generator
    gen_backbone_name = backbone.lower().replace("_cbam", "")

    gen_public = AlbumentationsDataGenerator(TEST_PUBLIC_PATH, batch_size=BATCH_SIZE, img_size=IMG_SIZE, backbone=gen_backbone_name)
    gen_local  = AlbumentationsDataGenerator(TEST_LOCAL_PATH,  batch_size=BATCH_SIZE, img_size=IMG_SIZE, backbone=gen_backbone_name)

    y_true_public = get_true_labels_from_keras_generator(gen_public)
    y_true_local  = get_true_labels_from_keras_generator(gen_local)

    y_pred_public = np.argmax(model.predict(gen_public, verbose=0), axis=1)
    y_pred_local  = np.argmax(model.predict(gen_local, verbose=0), axis=1)

    # Ensure model is not None (safety check)
    if model is None:
        print(f"Error: Model '{backbone}' could not be loaded. Skipping metrics")
        continue

    # Metrics
    report_public = classification_report(y_true_public, y_pred_public, output_dict=True, zero_division=0)
    report_local  = classification_report(y_true_local, y_pred_local, output_dict=True, zero_division=0)

    acc_public = report_public["accuracy"]
    acc_local  = report_local["accuracy"]
    f1_public  = report_public["weighted avg"]["f1-score"]
    f1_local   = report_local["weighted avg"]["f1-score"]

    domain_gap_acc = acc_public - acc_local
    domain_gap_f1  = f1_public - f1_local

    results.append({
        "Model": backbone,
        "Accuracy_Public": acc_public,
        "Accuracy_Local": acc_local,
        "F1_Public": f1_public,
        "F1_Local": f1_local,
        "Δ Accuracy": domain_gap_acc,
        "Δ F1": domain_gap_f1,
    })

    # confusion matrix
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (y_true, y_pred, title) in zip(
        axes,
        [(y_true_public, y_pred_public, "Public Test"),
         (y_true_local, y_pred_local, "Local Test")]
    ):
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
        ax.set_title(f"{backbone} - {title}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
    plt.tight_layout()
    plt.show()

# save results
df_results = pd.DataFrame(results).sort_values("Δ F1", ascending=False)
df_results.to_csv(OUTPUT_PATH / "domain_gap_metrics.csv", index=False)
display(df_results)

# Δ F1 Plot
plt.figure(figsize=(10,6))
plt.bar(df_results["Model"], df_results["Δ F1"], color="salmon")
plt.title("Domain Gap (Δ F1)")
plt.ylabel("Δ F1-score (Public - Local)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
best_model = df_results.loc[df_results["Δ F1"].idxmin()]
print(f"""
MOST ROBUST MODEL (IN TERMS OF Δ F1):
- {best_model['Model']}
- F1 Público: {best_model['F1_Public']:.3f}
- F1 Local:   {best_model['F1_Local']:.3f}
- Δ F1:       {best_model['Δ F1']:.3f}
""")

# Evaluating YOLO models

In [ ]:
DATASET_TEST = Path("..data/dataset/dataset_YOLO/test")
PROJECT_DIR_PATH = Path("..data/models/results")

In [ ]:
MODEL_VARIANTS = ["n", "m", "l"]
RESULTS_SUMMARY = [] 

class_dirs = sorted([d.name for d in DATASET_TEST.iterdir() if d.is_dir()])
file_list = []
y_true = []

for idx, name in enumerate(class_dirs):
    images = [str(f) for f in (DATASET_TEST / name).rglob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    file_list.extend(images)
    y_true.extend([idx] * len(images))

print(f"Data is ready: {len(file_list)} images, {len(class_dirs)} classes.")

for v in MODEL_VARIANTS:
    model_path = PROJECT_DIR_PATH / f"YOLO11{v}_cls" / "weights" / "best.pt"
    
    if not model_path.exists():
        print(f"Skipping YOLO11{v}: File not found in {model_path}")
        continue

    print(f"\n🚀 Evaluating YOLO11{v}...")
    model = YOLO(model_path)
    
    start_time = time.time()
    results = model.predict(source=file_list, device=0, verbose=False)
    inference_time = (time.time() - start_time) / len(file_list) 
    
    y_pred = [r.probs.top1 for r in results]
    

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    kappa = cohen_kappa_score(y_true, y_pred)
    
    RESULTS_SUMMARY.append({
        "Model": f"YOLO11{v}",
        "Acc": acc,
        "F1-Score": f1,
        "Kappa": kappa,
        "Time/Img (s)": inference_time
    })
    

    print(f"--- Report {v.upper()} ---")
    print(classification_report(y_true, y_pred, target_names=class_dirs))

print("\n" + "="*60 + "\n📊 Model comparison\n" + "="*60)
df_results = pd.DataFrame(RESULTS_SUMMARY).set_index("Model")
print(df_results.round(4))